In [22]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import logging

logging.basicConfig(level=logging.INFO)
logger=logging.getLogger(__name__)

spark=SparkSession.builder.appName("starting_etl_ingestion").getOrCreate()

INFO:py4j.clientserver:Error while sending or receiving.
Traceback (most recent call last):
  File "c:\Users\chitranjan kumar\AppData\Local\Programs\Python\Python310\lib\site-packages\py4j\clientserver.py", line 527, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
INFO:py4j.clientserver:Closing down clientserver connection
INFO:root:Exception while sending command.
Traceback (most recent call last):
  File "c:\Users\chitranjan kumar\AppData\Local\Programs\Python\Python310\lib\site-packages\py4j\clientserver.py", line 527, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\chitranjan kumar\AppData\Local\Programs\Python\Python310\lib\site-

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
for partition  in 
    df=spark.read.option("inferSchema","true").parquet("s3://e-commerce-sales-etl/bronze_data/orderss_data/")
    df.write.mode("overwrite").parquet("s3://e-commerce-sales-etl/silver_data/order_data/")

AnalysisException: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for Parquet. It must be specified manually.

In [ ]:
#code for order_table

In [2]:
from pyspark.sql.types import *
import boto3




s3=boto3.client("s3")
bucket="e-commerce-sales-etl"
bronze_table_prefix="bronze_data/orderss_data"
iam_role="arn:aws:iam::455303856855:role/redshift-s3_iam_role"
list_object=s3.list_objects_v2(
        Bucket=bucket,
        Prefix=bronze_table_prefix,
    
)
print(list_object)

bronze_partition=set()
for obj in list_object["Contents"]:
    key=obj["Key"]
    folder=key.split("/")[2]
    bronze_partition.add(folder)
bronze_partition=sorted(bronze_partition)


print(bronze_partition)




INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


{'ResponseMetadata': {'RequestId': 'RXZBVKBS5ZZK3FJJ', 'HostId': 'R0EbodsEmkfmkujsQrcwXSGXYuJY8VPEw3y1n0cWi/+Yd4Y0Ug5AdCEO5FLSgUgUDGx3/XKmlFo=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'R0EbodsEmkfmkujsQrcwXSGXYuJY8VPEw3y1n0cWi/+Yd4Y0Ug5AdCEO5FLSgUgUDGx3/XKmlFo=', 'x-amz-request-id': 'RXZBVKBS5ZZK3FJJ', 'date': 'Sat, 15 Aug 2026 05:01:01 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'bronze_data/orderss_data/20260727_060547/part-00000-10b64181-08bb-4d69-b9aa-f8eaedeb5905-c000.snappy.parquet', 'LastModified': datetime.datetime(2026, 7, 27, 6, 12, tzinfo=tzutc()), 'ETag': '"77789aeedc2c4d901f4bb1083c651368"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 3053, 'StorageClass': 'STANDARD'}, {'Key': 'bronze_data/orderss_data/20260727_060547/part-00001-10b64181-08bb-4d69-b9aa-f8eaedeb5905-c000.snappy.pa

In [3]:
from datetime import datetime
timestamp=datetime.now().strftime

In [5]:


silver_data_prefix="silver_data/order_data"

unprocessed_partition=s3.list_objects_v2(
    Bucket=bucket,
    Prefix=silver_data_prefix
)

silver_partition=set()
for object in unprocessed_partition["Contents"]:
    key=object["Key"]
    if key=="silver_data/order_data/": # we have to skip this the fisrt key as it doesn't have any parition key
        continue
    partition=key.split('/')[2]
    silver_partition.add(partition)
silver_partition=sorted(silver_partition)[-1]

print(silver_partition)


        
        


20260727_060547


In [6]:
pending_parition=[]
if not silver_partition:
    pending_parition.extend(bronze_partition)
else:
    for partition in bronze_partition:
        if partition>silver_partition:
            pending_parition.append(partition)
print(pending_parition)        

['20260728_024832']


In [16]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("payment_id", IntegerType(), True),
    StructField("status_id", IntegerType(), True),
    StructField("shipping_city", StringType(), True),
    StructField("shipping_state", StringType(), True),
    StructField("shipping_address", StringType(), True),
    StructField("order_total", DoubleType(), True),
    StructField("last_updated_timestamp", StringType(), True)
])

print(orders_schema.fields)



[StructField('order_id', IntegerType(), True), StructField('customer_id', IntegerType(), True), StructField('order_date', StringType(), True), StructField('payment_id', IntegerType(), True), StructField('status_id', IntegerType(), True), StructField('shipping_city', StringType(), True), StructField('shipping_state', StringType(), True), StructField('shipping_address', StringType(), True), StructField('order_total', DoubleType(), True), StructField('last_updated_timestamp', StringType(), True)]


In [17]:

def data_validation(s3_path:str):
    df=spark.read.option("inferSchema","true").parquet(s3_path)
    validation_results={
    "missing_field":[],
    "extra_field":[],
    #"datatype_mismatch":[],
    "nullability_mismatch":[]
    }
    for field in orders_schema:
        if field.name not in df.columns:
            validation_results["missing_field"].append(field.name)
        else:
            expected_type=field.dataType
            actual_type=df.schema[field.name].dataType
            expected_nullable_value=field.nullable
            actual_nullable_value=df.schema[field.name].nullable
            """if  expected_type!=actual_type:
                validation_results["datatype_mismatch"].append({
                    "field_name":field.name,
                    "expected_type":expected_type,
                    "actual_type":actual_type

                })"""
            if expected_nullable_value!=actual_nullable_value:
                validation_results["nullability_mismatch"].append({
                    "field_name":field.name,
                    "expected_nullable_value":expected_nullable_value,
                    "actual_nullable_value":actual_nullable_value

                })



  

    for field in df.columns:
        if field not in orders_schema.fieldNames():
            validation_results['extra_field'].append(field)
    print(validation_results)        
   
    
    
    #for column in orders_schema:
    

for partition in pending_parition:

    s3_path=f"s3://{bucket}/{bronze_table_prefix}/{partition}" 
    load_bronze_data=data_validation(s3_path)   

{'missing_field': [], 'extra_field': [], 'nullability_mismatch': []}


In [ ]:
def run_dqc(df):
    



    
    


# Customer_table

In [12]:
s3=boto3.client("s3")
bucket="e-commerce-sales-etl"
bronze_customer_prefix="bronze_data/customer_data"
iam_role="arn:aws:iam::455303856855:role/redshift-s3_iam_role"
list_object=s3.list_objects_v2(
        Bucket=bucket,
        Prefix=bronze_customer_prefix,
    
)
print(list_object)

bronze_partition=set()
for obj in list_object["Contents"]:
    key=obj["Key"]
    if key =='bronze_data/customer_data/':
        continue
    folder=key.split("/")[2]
    bronze_partition.add(folder)
bronze_partition=sorted(bronze_partition)


print(bronze_partition)


{'ResponseMetadata': {'RequestId': 'D5F6J738TF03CVQT', 'HostId': 'Npg7rJ0nONXXlCdZ8HV6Kq3TY/zmLMBqX1wBZjN8JBxhVTv+VkGBk7aLIJgv3/DJejlrt5l4MQ8=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'Npg7rJ0nONXXlCdZ8HV6Kq3TY/zmLMBqX1wBZjN8JBxhVTv+VkGBk7aLIJgv3/DJejlrt5l4MQ8=', 'x-amz-request-id': 'D5F6J738TF03CVQT', 'date': 'Sat, 15 Aug 2026 05:04:59 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'bronze_data/customer_data/', 'LastModified': datetime.datetime(2026, 7, 27, 5, 33, 19, tzinfo=tzutc()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'bronze_data/customer_data/20260727_060547/part-00000-e3913d94-ef60-4ae2-aeaa-d767cc42458d-c000.snappy.parquet', 'LastModified': datetime.datetime(2026, 7, 27, 6, 10, 50, tzinfo=tzutc()

In [40]:
df=spark.read.parquet("s3://e-commerce-sales-etl/bronze_data/customer_data/20260727_060547/part-00000-e3913d94-ef60-4ae2-aeaa-d767cc42458d-c000.snappy.parquet")
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- customer_segment_id: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- last_updated_timestamp: timestamp (nullable = true)
 |-- is_active: string (nullable = true)



In [13]:


silver_customer_prefix="silver_data/customer_data"

unprocessed_partition=s3.list_objects_v2(
    Bucket=bucket,
    Prefix=silver_customer_prefix
)
print(unprocessed_partition)
silver_partition=set()
for object in unprocessed_partition["Contents"]:
    key=object["Key"]
    print(key)
    if key=="silver_data/customer_data/": # we have to skip this the fisrt key as it doesn't have any parition key
        continue
    part=key.split("/")
    print(part)
    if len(part)<3 or not part[2]:
         continue
    partition=part[2]
    silver_partition.add(partition)
if silver_partition:

    latest_partition=sorted(silver_partition)[-1]

else:
    latest_partition=None
print(latest_partition)    


        
        


{'ResponseMetadata': {'RequestId': 'R7Z30VGKP08VJFPG', 'HostId': 'P6dAXT8YUxvT7DbrKN+WfXnzLLEd1IENELmvEomtB8qBWT2p/ZSUqZUGfDt7Ro+5XyYXXo5fBQE=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'P6dAXT8YUxvT7DbrKN+WfXnzLLEd1IENELmvEomtB8qBWT2p/ZSUqZUGfDt7Ro+5XyYXXo5fBQE=', 'x-amz-request-id': 'R7Z30VGKP08VJFPG', 'date': 'Sat, 15 Aug 2026 05:05:03 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'silver_data/customer_data/', 'LastModified': datetime.datetime(2026, 8, 11, 4, 43, 27, tzinfo=tzutc()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'silver_data/customer_data/2026-08-13/part-00000-29600407-6be5-4c62-90d2-624d4943d8a2-c000.snappy.parquet', 'LastModified': datetime.datetime(2026, 8, 13, 9, 42, 25, tzinfo=tzutc()), 'E

In [14]:
pending_parition=[]
if not latest_partition:
    pending_parition.extend(bronze_partition)
else:
    for partition in bronze_partition:
        if partition>latest_partition:
            pending_parition.append(partition)
print(pending_parition)        

['20260727_060547', '20260728_024832']


In [15]:
from pyspark.sql.types import *
customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("dob", DateType(), True),
    StructField("customer_segment_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("registration_date", DateType(), True),
    StructField("last_updated_timestamp", TimestampType(), True),
    StructField("is_active", StringType(), True)
])

    


In [17]:

def data_validation(s3_path:str):
    df=spark.read.option("inferSchema","true").parquet(s3_path)
    validation_results={
    "missing_field":[],
    "extra_field":[],
    #"datatype_mismatch":[],
    "nullability_mismatch":[]
    }
    for field in customer_schema:
        if field.name not in df.columns:
            validation_results["missing_field"].append(field.name)
        else:
            expected_type=field.dataType
            actual_type=df.schema[field.name].dataType
            expected_nullable_value=field.nullable
            actual_nullable_value=df.schema[field.name].nullable
            """if  expected_type!=actual_type:
                validation_results["datatype_mismatch"].append({
                    "field_name":field.name,
                    "expected_type":expected_type,
                    "actual_type":actual_type

                })"""
            if expected_nullable_value!=actual_nullable_value:
                validation_results["nullability_mismatch"].append({
                    "field_name":field.name,
                    "expected_nullable_value":expected_nullable_value,
                    "actual_nullable_value":actual_nullable_value

                })



  

    for field in df.columns:
        if field not in customer_schema.fieldNames():
            validation_results['extra_field'].append(field)
    print(validation_results)        
   
    
    
    #for column in orders_schema:
    

for partition in pending_parition:

    s3_path=f"s3://{bucket}/{bronze_customer_prefix}/{partition}" 
    load_bronze_data=data_validation(s3_path)   

Py4JJavaError: An error occurred while calling o47.parquet.
: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3581)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:782)
	at scala.collection.immutable.List.map(List.scala:236)
	at scala.collection.immutable.List.map(List.scala:79)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:780)
	at org.apache.spar

In [18]:
#DQ Check :(Note :for this please explore some apps like great expectation to expediate this work        )
from pyspark.sql.functions import length
df=spark.read.option("inferSchema","true").parquet(s3_path)
df.show()

#primary key validation:
df_checks=df.filter(col("customer_id").isNotNull())

#Remove duplicate customer_id
df_filter=df_checks.groupby("customer_id").count().withColumnRenamed("count", "customer_id_count")
df_dedup=df_filter.filter(col("customer_id_count")==1)


df_checks=df_checks.join(df_dedup,"customer_id","inner")

#Customer segment Validation

df_checks=df_checks.join(customer_segment_df,"customer_segment_id","inner").drop(customer_segment_df["is_active"])




#First name validation
df_checks=df_checks.filter(col("first_name").isNotNull())


#Last name validation

df_checks=df_checks.filter(col("last_name").isNotNull())

#Email deduplication

df_filter=df_checks.groupby("email").count().withColumnRenamed("count", "email_count")
df_dedup=df_filter.filter(col("email_count")==1)


df_checks=df_checks.join(df_dedup,"email","inner")
#Email validation

df_checks=df_checks.filter(col("email").isNotNull())


#Phone validation

df_checks=df_checks.filter(length(col("phone_number"))==10)

df_final=df_checks.select(
    col("email"),
    col("customer_segment_id"),
    col("customer_id"),
    col("first_name"),
    col("last_name"),
    col("phone_number"),
    col("gender"),
    col("dob"),
    col("city"),
    col("state"),
    col("country"),
    col("registration_date"),
    col("last_updated_timestamp"),
    col("is_active"),
    col("customer_id_count"),
    col("segment_name"),
    col("discount_percentage"),
    col("loyalty_points_multiplier"),
    col("description"),
    col("email_count")
)            

df_checks.show()

from datetime import datetime
today_date=datetime.now().strftime("%Y-%m-%-d")
s3_target_path=f"s3://e-commerce-sales-etl/silver_data/customer_data/{today_date}"
df_final.write.mode("overwrite").parquet(s3_target_path)
 








Py4JJavaError: An error occurred while calling o53.parquet.
: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3581)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:782)
	at scala.collection.immutable.List.map(List.scala:236)
	at scala.collection.immutable.List.map(List.scala:79)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:780)
	at org.apache.spar

In [31]:
from datetime import datetime
today_date=datetime.now().strftime("%Y-%m-%-d")
print(today_date)

2026-08-13


In [11]:
import os
#Customer segment Validation
mysql_config={
        
        "host": os.getenv("host", "host.docker.internal"),
        "port": os.getenv("port", "3306"),
        "database": os.getenv("database", "ecommerce"),
        "username": os.getenv("username","root"),
        "password": os.getenv("password","Chiku@123"),
        "driver": os.getenv("driver", "com.mysql.cj.jdbc.Driver")
    }
url=f"jdbc:mysql://{mysql_config['host']}:{mysql_config['port']}/{mysql_config['database']}"
table="customer_segment_lookup"
customer_segment_df=spark.read.jdbc(
    url=url,
    table=table,
    properties={
    "user":mysql_config["username"],
    "password":mysql_config["password"],
    "driver":mysql_config["driver"]
    }
    
    )




customer_segment_df.show()

+-------------------+----------------+-------------------+-------------------------+--------------------+---------+
|customer_segment_id|    segment_name|discount_percentage|loyalty_points_multiplier|         description|is_active|
+-------------------+----------------+-------------------+-------------------------+--------------------+---------+
|                  1|         Regular|               0.00|                     1.00|    Regular Customer|        Y|
|                  2|          Silver|               2.50|                     1.20|Silver Loyalty Cu...|        Y|
|                  3|            Gold|               5.00|                     1.50|Gold Loyalty Cust...|        Y|
|                  4|         Premium|               8.00|                     2.00|    Premium Customer|        Y|
|                  5|             VIP|              10.00|                     3.00|        VIP Customer|        Y|
|                  6|       Corporate|              12.00|              

# Product silver Table

In [ ]:


s3://e-commerce-sales-etl/silver_data/product_data/

In [3]:
s3=boto3.client("s3")
bucket="e-commerce-sales-etl"
bronze_product_prefix="bronze_data/product_data"
iam_role="arn:aws:iam::455303856855:role/redshift-s3_iam_role"
list_object=s3.list_objects_v2(
        Bucket=bucket,
        Prefix=bronze_product_prefix,
    
)
print(list_object)

bronze_partition=set()
for obj in list_object["Contents"]:
    key=obj["Key"]
    if key =='bronze_data/product_data/':
        continue
    folder=key.split("/")[2]
    bronze_partition.add(folder)
bronze_partition=sorted(bronze_partition)


print(bronze_partition)


{'ResponseMetadata': {'RequestId': '1410ASPZGFVY1P7X', 'HostId': 'oPg/BertqQ4tzxVG4gLlQ5gyZl04nGB+TlQeNe+Blr+3JXK4f/1lFjtolH9KKa3y3Qkyk1wz9qlUHE+juCYGBEDTLkh/CbIH', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'oPg/BertqQ4tzxVG4gLlQ5gyZl04nGB+TlQeNe+Blr+3JXK4f/1lFjtolH9KKa3y3Qkyk1wz9qlUHE+juCYGBEDTLkh/CbIH', 'x-amz-request-id': '1410ASPZGFVY1P7X', 'date': 'Sat, 15 Aug 2026 05:01:34 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'bronze_data/product_data/', 'LastModified': datetime.datetime(2026, 7, 27, 5, 33, 32, tzinfo=tzutc()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'bronze_data/product_data/20260727_060547/part-00000-f7117eb4-afe3-46db-b47f-b92dc1e0e8cc-c000.snappy.parquet', 'LastModified': datetime.datetime(

In [5]:


silver_product_prefix="silver_data/product_data"

unprocessed_partition=s3.list_objects_v2(
    Bucket=bucket,
    Prefix=silver_product_prefix
)
print(unprocessed_partition)
silver_partition=set()
for object in unprocessed_partition["Contents"]:
    key=object["Key"]
    print(key)
    if key=="silver_data/product_data/": # we have to skip this the fisrt key as it doesn't have any parition key
        continue
    part=key.split("/")
    print(part)
    if len(part)<3 or not part[2]:
         continue
    partition=part[2]
    silver_partition.add(partition)
if silver_partition:

    latest_partition=sorted(silver_partition)[-1]

else:
    latest_partition=None
print(latest_partition)    


        
        


{'ResponseMetadata': {'RequestId': 'C86C9S7PS2YZZ1VS', 'HostId': 'KP8eM51j2N1Edq4nxhuU63lJayo/Spj/Jn7VToMRaUZNbgUpW1pY+hMEAYbUtm/WI0JehTxuNG0=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'KP8eM51j2N1Edq4nxhuU63lJayo/Spj/Jn7VToMRaUZNbgUpW1pY+hMEAYbUtm/WI0JehTxuNG0=', 'x-amz-request-id': 'C86C9S7PS2YZZ1VS', 'date': 'Sat, 15 Aug 2026 05:01:53 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'silver_data/product_data/', 'LastModified': datetime.datetime(2026, 8, 13, 10, 1, 33, tzinfo=tzutc()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 0, 'StorageClass': 'STANDARD'}], 'Name': 'e-commerce-sales-etl', 'Prefix': 'silver_data/product_data', 'MaxKeys': 1000, 'EncodingType': 'url', 'KeyCount': 1}
silver_data/product_data/
None


In [7]:
pending_parition=[]
if not latest_partition:
    pending_parition.extend(bronze_partition)
else:
    for partition in bronze_partition:
        if partition>latest_partition:
            pending_parition.append(partition)
print(pending_parition)        

['20260727_060547', '20260728_024832']


In [9]:
from pyspark.sql.types import *

product_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("category_id", IntegerType(), True),
    StructField("unit_price", DecimalType(10, 2), True),
    StructField("manufacturing_date", DateType(), True),
    StructField("expiry_date", DateType(), True),
    StructField("weight_grams", DecimalType(8, 2), True),
    StructField("color", StringType(), True),
    StructField("is_active", StringType(), True),
    StructField("last_updated_timestamp", TimestampType(), True)
])


In [11]:

def data_validation(s3_path:str):
    df=spark.read.option("inferSchema","true").parquet(s3_path)
    validation_results={
    "missing_field":[],
    "extra_field":[],
    #"datatype_mismatch":[],
    "nullability_mismatch":[]
    }
    for field in product_schema:
        if field.name not in df.columns:
            validation_results["missing_field"].append(field.name)
        else:
            expected_type=field.dataType
            actual_type=df.schema[field.name].dataType
            expected_nullable_value=field.nullable
            actual_nullable_value=df.schema[field.name].nullable
            """if  expected_type!=actual_type:
                validation_results["datatype_mismatch"].append({
                    "field_name":field.name,
                    "expected_type":expected_type,
                    "actual_type":actual_type

                })"""
            if expected_nullable_value!=actual_nullable_value:
                validation_results["nullability_mismatch"].append({
                    "field_name":field.name,
                    "expected_nullable_value":expected_nullable_value,
                    "actual_nullable_value":actual_nullable_value

                })



  

    for field in df.columns:
        if field not in product_schema.fieldNames():
            validation_results['extra_field'].append(field)
    print(validation_results)        
   
    
    
    #for column in orders_schema:
    

for partition in pending_parition:

    s3_path=f"s3://{bucket}/{bronze_product_prefix}/{partition}" 
    load_bronze_data=data_validation(s3_path)   

Py4JJavaError: An error occurred while calling o35.parquet.
: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3581)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:782)
	at scala.collection.immutable.List.map(List.scala:236)
	at scala.collection.immutable.List.map(List.scala:79)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:780)
	at org.apache.spar

In [21]:
#DQ Check :(Note :for this please explore some apps like great expectation to expediate this work        )
from pyspark.sql.functions import length,trim,col
df=spark.read.option("inferSchema","true").parquet(s3_path)
df.show()

#primary key validation:
df_checks=df.filter(col("product_id").isNotNull())

#Remove duplicate product_id
df_filter=df_checks.groupby("product_id").count().withColumnRenamed("count", "product_id_count")
df_dedup=df_filter.filter(col("product_id_count")==1)

df_checks=df_checks.join(df_dedup,"product_id","inner")
#product_name validation and standardization

df_checks=df_checks.filter(col("product_name").isNotNull())
df_checks=df_checks.withColumn("product_name",trim(col("product_name")))

#Brand Column validation:
df_checks=df_checks.filter(col("brand").isNotNull())
#Customer segment Validation

df_checks=df_checks.join(category_lookup,"category_id","inner").drop(category_lookup["is_active"])



#unit price validation
df_checks=df_checks.filter((col("unit_price").isNotNull()) & (col("unit_price") > 0))

#manufacturing_date validation
df_checks=df_checks.filter((col("manufacturing_date").isNotNull()) & (col("expiry_date") > col("manufacturing_date")))

#weight gram validation
df_checks=df_checks.filter(col("weight_grams")>0)

#Color gram validation

#Email deduplication

df_filter=df_checks.groupby("email").count().withColumnRenamed("count", "email_count")
df_dedup=df_filter.filter(col("email_count")==1)


df_checks=df_checks.join(df_dedup,"email","inner")
#Email validation

df_checks=df_checks.filter(col("email").isNotNull())



#Phone validation

df_checks=df_checks.filter(length(col("phone_number"))==10)

df_final=df_checks.select(
    col("email"),
    col("customer_segment_id"),
    col("customer_id"),
    col("first_name"),
    col("last_name"),
    col("phone_number"),
    col("gender"),
    col("dob"),
    col("city"),
    col("state"),
    col("country"),
    col("registration_date"),
    col("last_updated_timestamp"),
    col("is_active"),
    col("customer_id_count"),
    col("segment_name"),
    col("discount_percentage"),
    col("loyalty_points_multiplier"),
    col("description"),
    col("email_count")
)            

df_checks.show()

from datetime import datetime
today_date=datetime.now().strftime("%Y-%m-%-d")
s3_target_path=f"s3://e-commerce-sales-etl/silver_data/customer_data/{today_date}"
df_final.write.mode("overwrite").parquet(s3_target_path)

Py4JJavaError: An error occurred while calling o71.parquet.
: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3581)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:782)
	at scala.collection.immutable.List.map(List.scala:236)
	at scala.collection.immutable.List.map(List.scala:79)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:780)
	at org.apache.spar

In [32]:
import os
#Customer segment Validation
mysql_config={
        
        "host": os.getenv("host", "host.docker.internal"),
        "port": os.getenv("port", "3306"),
        "database": os.getenv("database", "ecommerce"),
        "username": os.getenv("username","root"),
        "password": os.getenv("password","Chiku@123"),
        "driver": os.getenv("driver", "com.mysql.cj.jdbc.Driver")
    }
url=f"jdbc:mysql://{mysql_config['host']}:{mysql_config['port']}/{mysql_config['database']}"
table="category_lookup"
customer_segment_df=spark.read.jdbc(
    url=url,
    table=table,
    properties={
    "user":mysql_config["username"],
    "password":mysql_config["password"],
    "driver":mysql_config["driver"]
    }
    
    )




customer_segment_df.show()

+-----------+---------------+-----------+--------------+-----------+---------+
|category_id|  category_name| department|gst_percentage|return_days|is_active|
+-----------+---------------+-----------+--------------+-----------+---------+
|          1|  Mobile Phones|Electronics|         18.00|          7|        Y|
|          2|     Headphones|Electronics|         18.00|         10|        Y|
|          3|        Laptops|  Computers|         18.00|         15|        Y|
|          4|       Monitors|  Computers|         18.00|         15|        Y|
|          5|    Accessories|  Computers|         18.00|          7|        Y|
|          6|       Printers|     Office|         18.00|         10|        Y|
|          7|        Tablets|Electronics|         18.00|         10|        Y|
|          8|  Smart Watches|Electronics|         18.00|          7|        Y|
|          9|        Cameras|Electronics|         18.00|         15|        Y|
|         10|       Speakers|Electronics|         18

In [ ]:
import os
from datetime import datetime


bucket="e-commerce-sales-etl"
prefix="ingestion_data"
base_path=f"s3://{bucket}/{prefix}"
bronze_prefix="bronze_data"
bronze_path=f"s3://{bucket}/{bronze_prefix}"
timestamp=datetime.now().strftime("%Y%m%d_%H%M%S")


table_name=[{
            "source_type":"my_sql",
            "source_table":"customers",
            "target_table":"customer_data"
            },
            {
            "source_type":"my_sql",
            "source_table":"products",
            "target_table":"product_data"
            },
            {"source_type":"s3",
            "source_table":"inventory_data",
            "target_table":"inventory_data"
            },
            {"source_type":"s3",
            "source_table":"order_item_data",
            "target_table":"order_item_data"
                },
            {"source_type":"s3",
            "source_table":"orders_data",
            "target_table":"orderss_data"
            }
             ]

for table in table_name:
        if table["source_type"]=="my_sql":

            target_table=table["target_table"]
            s3_bronze_path=f"{bronze_path}/{target_table}/{timestamp}"
            print(s3_bronze_path)

"""for table in table_name:
        if table["source_type"]=="my_sql":
            print(f"{table['source_table']}")
        elif table["source_type"]=="s3":
            s3_path=f"{base_path}/{table['source_table']}"
            df=spark.read.option("multiline","True").json(s3_path)
            df.show()"""